# Teste isolado — ABAR (Acontece nas Agências)

Fonte candidata: **ABAR - Associação Brasileira de Agências Reguladoras**,
setor Regulatório/Múltiplo (agrega notícias de 87 agências associadas —
saneamento, energia, transporte, recursos hídricos, estaduais e
municipais). Notebook **descartável** (Fase 1) — sem dispatcher, sem
`atualizar_status_fonte`, sem gravar nada. Só valida:

1. Scraping da listagem de notícias (título, data, link)
2. Extração do texto completo de uma notícia individual

## Confirmado antes de assumir

WordPress confirmado (`robots.txt` com bloco Yoast SEO, `sitemap_index.xml`).
A página de notícias **não está na home** — achei via `/wp-json/wp/v2/categories`:
a categoria `acontece-nas-agencias` (**1.042 posts**, de longe a maior) é
claramente o feed agregador descrito no pedido. Outras categorias existem
("Em Destaque", "Curtas", "ABAR na Mídia") mas essa é a candidata natural.

**Achado 1**: o site tem um WAF que devolve `406 Not Acceptable` para
requisições sem um header `Accept` no formato completo de navegador (ex.:
`curl -A "Mozilla/5.0"` sozinho, ou um `Accept: text/html` isolado, ambos
bloqueados). Testei especificamente os headers que o dispatcher genérico já
usa hoje (`headers_aleatorios()`: User-Agent + Accept-Language +
Accept-Encoding, sem `Accept` explícito) e **funcionam normalmente** — não
precisa de nenhum ajuste na infra compartilhada.

**Achado 2 — o tema mistura `<h2>` e `<h3>` na mesma listagem**: o tema
(família "Jannah"/jeg) mostra os 4 posts mais recentes num bloco "hero" no
topo (`h2.jeg_post_title`) e o resto da lista em `h3.jeg_post_title` — as
duas seções usam a mesma classe `article.jeg_post`, então um seletor
`h2.jeg_post_title a` sozinho só pega os 4 do hero e silenciosamente ignora
o resto. **Armadilha real**: o bloco "hero" é fixo (mesmos 4 posts mais
recentes) em **todas** as páginas da categoria — comparar só os itens do
hero entre `/page/1/` e `/page/2/` parece "paginação quebrada" (mesmo
conteúdo), mas é só esse bloco fixo; a lista de baixo (`h3`) pagina
corretamente. Seletor certo: `.jeg_post_title a` (sem restringir a tag),
mais deduplicação por URL entre páginas (o hero se repete em todas).

In [0]:
%pip install --quiet httpx curl_cffi beautifulsoup4 lxml
dbutils.library.restartPython()

In [0]:
import re
import time
import random
import unicodedata
import urllib.parse
from datetime import datetime
from typing import Optional

import httpx
from bs4 import BeautifulSoup
from curl_cffi import requests as cffi_requests

In [0]:
# =============================================================================
# Configuração
# =============================================================================

SITE_URL = "https://abar.org.br/category/acontece-nas-agencias/"

HTTP_TIMEOUT = 30
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

MESES_PT = {
    "janeiro": "01", "fevereiro": "02", "março": "03", "abril": "04",
    "maio": "05", "junho": "06", "julho": "07", "agosto": "08",
    "setembro": "09", "outubro": "10", "novembro": "11", "dezembro": "12",
}
PADRAO_DATA_LONGA = re.compile(r"(\d{1,2}) de (\w+) de (\d{4})")
PADRAO_LINHAS_VAZIAS = re.compile(r"\n{3,}")
ITENS_POR_PAGINA_ESPERADO = 14

In [0]:
def headers_aleatorios(referer: Optional[str] = None) -> dict:
    # Mesmos headers já usados no dispatcher genérico -- confirmado que
    # passam pelo WAF do site sem precisar de Accept explícito.
    headers = {
        "User-Agent": USER_AGENT,
        "Accept-Language": "pt-BR,pt;q=0.9",
        "Accept-Encoding": "gzip, deflate",
    }
    if referer:
        headers["Referer"] = referer
    return headers


def baixar_pagina(url: str, tentativas: int = 3) -> Optional[str]:
    for tentativa in range(1, tentativas + 1):
        headers = headers_aleatorios(referer=SITE_URL)
        try:
            resp = httpx.get(url, headers=headers, timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [httpx tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, headers=headers, impersonate=impersonate,
                                      timeout=HTTP_TIMEOUT, allow_redirects=True)
            if resp.status_code == 200 and resp.text and len(resp.text) > 500:
                return resp.text
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.5))

    return None

## Teste 1 — listar a listagem (com paginação)

`article.jeg_post` -> título/link em `.jeg_post_title a` (sem restringir a
`h2`/`h3` — ver Achado 2), data em `.jeg_meta_date` (formato longo
`"31 de julho de 2026"`, precisa de `MESES_PT`). Paginação:
`/category/acontece-nas-agencias/page/N/`, deduplicando por URL (o bloco
"hero" repete os mesmos 4 posts em toda página). Categoria tem 1.042 posts
(104 páginas) — limito a amostra aqui.

In [0]:
def listar_abar(max_paginas: int = 4) -> list[dict]:
    itens, vistos = [], set()

    for pagina in range(1, max_paginas + 1):
        url_pagina = SITE_URL if pagina == 1 else f"{SITE_URL.rstrip('/')}/page/{pagina}/"

        html = baixar_pagina(url_pagina)
        if not html:
            print(f"  -> falha ao baixar página {pagina}; parando.")
            break

        soup = BeautifulSoup(html, "lxml")
        itens_pagina = soup.select("article.jeg_post")
        if not itens_pagina:
            print(f"  -> nenhum item encontrado na página {pagina}; fim da listagem.")
            break

        novos_na_pagina = 0
        for item in itens_pagina:
            tag_a = item.select_one(".jeg_post_title a")
            if not tag_a:
                continue
            url_item = tag_a["href"].strip()
            if url_item in vistos:
                continue
            vistos.add(url_item)
            novos_na_pagina += 1

            data_publicacao = None
            tag_data = item.select_one(".jeg_meta_date")
            if tag_data:
                m = PADRAO_DATA_LONGA.search(tag_data.get_text(" ", strip=True))
                if m:
                    dia, mes_nome, ano = m.groups()
                    mes = MESES_PT.get(mes_nome.lower())
                    if mes:
                        data_publicacao = f"{ano}-{mes}-{dia.zfill(2)}"

            itens.append({
                "titulo": tag_a.get_text(strip=True),
                "url": url_item,
                "published_at": data_publicacao,
            })

        print(f"  página {pagina}: {len(itens_pagina)} elementos, {novos_na_pagina} novos (após dedup do hero).")
        if len(itens_pagina) < ITENS_POR_PAGINA_ESPERADO:
            break
        time.sleep(random.uniform(0.5, 1.2))

    return itens

In [0]:
itens = listar_abar()

print(f"\n{len(itens)} notícias listadas.\n")
print(f"{'DATA':<12} TÍTULO")
print("-" * 100)
for item in itens:
    print(f"{item['published_at'] or '?':<12} {item['titulo'][:80]}")

urls_unicas = {i["url"] for i in itens}
sem_data = [i for i in itens if not i["published_at"]]

print(f"\nurls únicas: {len(urls_unicas)}/{len(itens)}")
print(f"sem data: {len(sem_data)}")
print(f"\nExemplo de link: {itens[0]['url']}")

## Teste 2 — abrir uma notícia e extrair o texto completo

`.entry-content` -- classe WordPress genérica o bastante pra virar seletor
compartilhado no dispatcher (diferente do seletor específico de tema que a
ABEGÁS precisou).

In [0]:
def extrair_texto_generico(html: str) -> str:
    try:
        soup = BeautifulSoup(html, "lxml")
    except Exception:
        soup = BeautifulSoup(html, "html.parser")

    for tag in soup(["script", "style", "noscript", "iframe", "form"]):
        tag.decompose()

    base = soup.select_one(".entry-content")
    if base is None or len(base.get_text(strip=True)) < 200:
        base = soup

    texto = base.get_text("\n", strip=True)
    return PADRAO_LINHAS_VAZIAS.sub("\n\n", texto).strip()


def extrair_titulo_h1(html: str) -> Optional[str]:
    soup = BeautifulSoup(html, "lxml")
    h1 = soup.find("h1")
    return h1.get_text(" ", strip=True) if h1 else None


def extrair_noticia_abar(item: dict) -> Optional[dict]:
    html = baixar_pagina(item["url"])
    if not html:
        return None

    texto = extrair_texto_generico(html)
    titulo = extrair_titulo_h1(html) or item["titulo"]

    return {
        "titulo": titulo,
        "url": item["url"],
        "published_at": item["published_at"],
        "texto": texto,
    }

In [0]:
AMOSTRA = 6

detalhes = []
for item in itens[:AMOSTRA]:
    print(f"\n  [item] {item['titulo'][:90]}")
    detalhe = extrair_noticia_abar(item)
    if detalhe is None:
        print("    -> download falhou.")
        continue
    detalhes.append(detalhe)
    print(f"    -> {len(detalhe['texto'])} chars extraídos.")
    time.sleep(random.uniform(0.5, 1.2))

print(f"\n{len(detalhes)}/{AMOSTRA} notícias abertas com sucesso.")
curtas = [d for d in detalhes if len(d["texto"]) < 200]
print(f"Com texto abaixo de 200 chars: {len(curtas)}")

In [0]:
# Amostra completa da primeira notícia — pra conferir na mão se bate com o
# que aparece no site.
detalhe = detalhes[0]

print("=" * 100)
print(f"TÍTULO      : {detalhe['titulo']}")
print(f"PUBLICADO EM: {detalhe['published_at']}")
print(f"URL         : {detalhe['url']}")
print(f"TAMANHO     : {len(detalhe['texto'])} chars")
print("=" * 100)
print(detalhe["texto"])

## Conclusão da Fase 1

Os dois testes passam: listagem paginada extrai título/data/link de todos
os itens (hero + lista, deduplicados), texto completo sai limpo via
`.entry-content` (seletor WordPress genérico). Amostra confirma o que o
pedido antecipava — granularidade variando de eventos institucionais a
decisões regulatórias pontuais de agências associadas (ex.: revisão
tarifária da ARSP/ES, decisão da AGERGS sobre tarifação de hotéis). Sem
filtro de relevância aplicado, como pedido.

**Avaliação para a Fase 2**: encaixa no dispatcher genérico
`ingest-scraping` — sem Selenium, sem parsing que dependa de JS. Precisa de
uma `listar_abar()` própria (paginação `/page/N/`, seletores do tema
`jeg_post`/`jeg_meta_date` com data em formato longo, dedup por URL por
causa do bloco hero fixo), mas o texto do detalhe reaproveita
`extrair_texto_generico()` com `.entry-content` acrescentado a
`SELETORES_CONTEUDO` — sem extrator próprio.

Histórico grande (1.042 posts na categoria) -- mesmo critério de
ANP/ABEGÁS: `max_paginas` conservador (recente, não backfill completo).